# Malaria Burden & Intervention Effectiveness Analysis
## Notebook 01 — Data Preparation & SQL Pipeline

**Author:** Stephen Udoh
**Date:** June 2026
**GitHub:** github.com/Stephen-Udoh/malaria-analysis

---

### Research Question
Does ITN coverage significantly reduce malaria incidence 
and mortality across sub-Saharan African countries? 
Which demographic and geographic factors moderate 
this relationship?

---

### Data Sources
| Source | Dataset | Years |
|--------|---------|-------|
| WHO GHO | Malaria burden — incidence & mortality | 2000–2024 |
| WHO GHO | Malaria interventions — ITN, IRS, ACT | 2015–2024 |
| World Bank | GDP per capita & urban population % | 2000–2024 |

---

### Notebook Structure
1. Environment setup and database connection
2. Load raw CSV files into MySQL
3. Audit tables in SQL
4. Clean and filter to sub-Saharan Africa
5. Transform World Bank data from wide to long
6. Convert counts to rates
7. Join all datasets into analytical tables
8. Export final datasets for analysis

---

### Analytical Plan
| Analysis | Period | Method | Tool |
|----------|--------|--------|------|
| Burden trend over time | 2000–2024 | Time series | Python |
| Intervention effectiveness | 2015–2024 | Regression | SPSS + Python |
| ITN scale-up comparison | 2015–2024 | T-test | SPSS + Python |
| Burden prediction model | 2015–2024 | Random Forest | Python ML |
| Malaria forecast to 2030 | 2000–2024 | Prophet | Python ML |

## 1. Environment Setup & Database Connection

In [3]:
# =============================================================
# IMPORTS
# All libraries used in this notebook declared in one place
# Add new libraries here rather than scattered across cells
# =============================================================

import pandas as pd                         # data manipulation
import numpy as np                          # numerical operations
from sqlalchemy import create_engine, text  # database connection
from dotenv import load_dotenv              # load credentials
import os                                   # access environment variables
import warnings                             # suppress minor warnings

warnings.filterwarnings('ignore')

print("✓ Libraries loaded successfully")

✓ Libraries loaded successfully


### 1.1 Load Credentials & Configure Connection
Credentials are stored in a `.env` file in the project root.
This file is excluded from version control via `.gitignore`.
Anyone reusing this notebook creates their own `.env` file.

In [4]:
# =============================================================
# LOAD CREDENTIALS FROM .env FILE
# Reads database credentials from local environment file
# Never hardcodes sensitive information in the notebook
# =============================================================

# Build path to .env file in project root
# Our notebook is in /notebooks/ so we go one level up
dotenv_path = os.path.join(os.path.dirname(os.getcwd()), '.env')

# Load the environment variables from .env file
load_dotenv(dotenv_path)

# Read each credential from environment
DB_CONFIG = {
    'user'    : os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
    'host'    : os.getenv('DB_HOST'),
    'port'    : os.getenv('DB_PORT'),
    'database': os.getenv('DB_NAME')
}

# Verify all credentials loaded — catches missing .env entries
missing = [key for key, val in DB_CONFIG.items() if val is None]

if missing:
    print(f"✗ Missing credentials: {missing}")
    print("  Check your .env file exists in project root")
else:
    print("✓ All credentials loaded from .env file")
    print(f"  User     : {DB_CONFIG['user']}")
    print(f"  Host     : {DB_CONFIG['host']}")
    print(f"  Port     : {DB_CONFIG['port']}")
    print(f"  Database : {DB_CONFIG['database']}")
    print("  Password : ********")

✓ All credentials loaded from .env file
  User     : root
  Host     : localhost
  Port     : 3306
  Database : malaria_project
  Password : ********


### 1.2 Database Connection

In [5]:
# Build connection string from loaded credentials
CONNECTION_STRING = (
    f"mysql+mysqlconnector://"
    f"{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/"
    f"{DB_CONFIG['database']}"
)

def create_db_engine(connection_string):
    """
    Creates and validates a SQLAlchemy engine.
    Returns engine object if successful, None if failed.
    """
    try:
        engine = create_engine(connection_string)
        with engine.connect() as conn:
            result = conn.execute(text("SELECT DATABASE()"))
            db_name = result.fetchone()[0]
            print(f"✓ Connected to: {db_name}")
        return engine
    except Exception as e:
        print(f"✗ Connection failed: {e}")
        return None

engine = create_db_engine(CONNECTION_STRING)

✓ Connected to: malaria_project


In [6]:
def run_query(query, engine):
    """
    Executes a SQL query and returns a pandas DataFrame.
    Used throughout this notebook for all SQL operations.
    """
    try:
        with engine.connect() as conn:
            return pd.read_sql(text(query), conn)
    except Exception as e:
        print(f"✗ Query failed: {e}")
        return None

## 2. Load Raw CSV Files into MySQL

Each WHO file is in long format — one row per country per year.
World Bank files are in wide format — years as columns.
All files are loaded as-is into raw tables first.
Cleaning and transformation happens in subsequent steps.

In [7]:
import os

# Path to raw data folder — one level up from notebooks
RAW_DATA_PATH = os.path.join(os.path.dirname(os.getcwd()), 'data', 'raw')

# Map each file to a meaningful table name in MySQL
# Keys = table names, Values = CSV filenames
FILES_TO_LOAD = {
    'raw_mortality'     : 'MALARIA_EST_MORTALITY.csv',
    'raw_incidence'     : 'MALARIA_EST_INCIDENCE.csv',
    'raw_itn'           : 'MALARIA_ITN_COVERAGE.csv',
    'raw_act'           : 'MALARIA_ACT_TREATED.csv',
    'raw_irs'           : 'MALARIA_IRS_COVERAGE.csv',
    'raw_gdp'           : 'API_NY.GDP.PCAP.CD_DS2_en_csv_v2_273495.csv',
    'raw_urban'         : 'API_SP.URB.TOTL.IN.ZS_DS2_en_csv_v2_276471.csv'
}

print(f"Data path: {RAW_DATA_PATH}")
print(f"Files to load: {len(FILES_TO_LOAD)}")

Data path: C:\Users\HP\Desktop\malaria_analysis\data\raw
Files to load: 7


In [10]:
def load_csv_to_mysql(files_dict, data_path, engine):
    """
    Loads multiple CSV files into MySQL as raw tables.
    
    Replaces table if it already exists — safe to re-run.
    Skips files that cannot be found and reports clearly.
    Returns a summary of what was loaded successfully.
    """
    summary = []

    for table_name, filename in files_dict.items():
        filepath = os.path.join(data_path, filename)

        # Check file exists before attempting load
        if not os.path.exists(filepath):
            print(f"✗ File not found: {filename}")
            summary.append({
                'table'  : table_name,
                'file'   : filename,
                'rows'   : 0,
                'status' : 'File not found'
            })
            continue

        try:
            # Read CSV into pandas
            df = pd.read_csv(filepath, encoding='utf-8')

            # Write to MySQL — replace if table exists
            df.to_sql(
                name      = table_name,
                con       = engine,
                if_exists = 'replace',
                index     = False
            )

            print(f"✓ {table_name:<20} {len(df):>6} rows loaded")
            summary.append({
                'table'  : table_name,
                'file'   : filename,
                'rows'   : len(df),
                'status' : 'Success'
            })

        except Exception as e:
            print(f"✗ {table_name} failed: {e}")
            summary.append({
                'table'  : table_name,
                'file'   : filename,
                'rows'   : 0,
                'status' : str(e)
            })

    # Return summary as DataFrame for easy review
    return pd.DataFrame(summary)


# Run the loader
print("Loading files into MySQL...\n")
load_summary = load_csv_to_mysql(FILES_TO_LOAD, RAW_DATA_PATH, engine)

print("\n--- Load Summary ---")
print(load_summary)

Loading files into MySQL...

✓ raw_mortality          2834 rows loaded
✓ raw_incidence          2855 rows loaded
✓ raw_itn                 400 rows loaded
✓ raw_act                 505 rows loaded
✓ raw_irs                 579 rows loaded
✓ raw_gdp                 266 rows loaded
✓ raw_urban               266 rows loaded

--- Load Summary ---
           table                                            file  rows  \
0  raw_mortality                       MALARIA_EST_MORTALITY.csv  2834   
1  raw_incidence                       MALARIA_EST_INCIDENCE.csv  2855   
2        raw_itn                        MALARIA_ITN_COVERAGE.csv   400   
3        raw_act                         MALARIA_ACT_TREATED.csv   505   
4        raw_irs                        MALARIA_IRS_COVERAGE.csv   579   
5        raw_gdp     API_NY.GDP.PCAP.CD_DS2_en_csv_v2_273495.csv   266   
6      raw_urban  API_SP.URB.TOTL.IN.ZS_DS2_en_csv_v2_276471.csv   266   

    status  
0  Success  
1  Success  
2  Success  
3  Success

## 3. Verify Tables in MySQL

Confirm all seven raw tables loaded correctly.
Check row counts match our audit log expectations.

In [11]:
# Verify all tables exist in the database
# Row counts should match the audit log exactly

verification_query = """
    SELECT 
        table_name,
        table_rows
    FROM information_schema.tables
    WHERE table_schema = 'malaria_project'
    ORDER BY table_name;
"""

tables = run_query(verification_query, engine)
print("Tables in malaria_project database:\n")
print(tables.to_string(index=False))

Tables in malaria_project database:

   TABLE_NAME  TABLE_ROWS
      raw_act         505
      raw_gdp         266
raw_incidence        2756
      raw_irs         579
      raw_itn         400
raw_mortality        2782
    raw_urban         266


In [12]:
# Preview first 3 rows of each table
# Confirms data loaded correctly and columns look right

tables_to_preview = [
    'raw_mortality',
    'raw_incidence', 
    'raw_itn',
    'raw_act',
    'raw_irs',
    'raw_gdp',
    'raw_urban'
]

for table in tables_to_preview:
    print(f"\n{'='*60}")
    print(f"Table: {table}")
    print('='*60)
    df = run_query(f"SELECT * FROM {table} LIMIT 3;", engine)
    print(df.to_string(index=False))


Table: raw_mortality
     Id         IndicatorCode SpatialDimension SpatialDimensionValueCode ParentLocationCode ParentLocation TimeDimension  TimeDim DisaggregatingDimension1 DisaggregatingDimension1ValueCode DisaggregatingDimension2 DisaggregatingDimension2ValueCode DisaggregatingDimension3 DisaggregatingDimension3ValueCode DataSourceDimension DataSourceDimensionValueCode               Value  NumericValue       Low      High Comments                 Date  TimeDimensionValue TimeDimensionBegin TimeDimensionEnd Unnamed: 25 Unnamed: 26
9418354 MALARIA_EST_MORTALITY          COUNTRY                       ECU                AMR       Americas          YEAR     2010                     None                              None                     None                              None                     None                              None                None                         None             0 [0-0]      0.000000  0.000000  0.000000     None 2025-12-19T14:27:58Z                201